# Lesson 05: The LLM proxy

**Goal:** map `provider:model` routes to provider classes and see what the proxy adds on top.

**Before you start**
- Theory: [lesson 05](../../docs/lessons/05_llm_proxy.md)
- Python concepts: inheritance from an `ABC`, `@abstractmethod`, decorators
- Files used: `src/docrag/llm/base.py`, `src/docrag/llm/proxy.py`, `src/docrag/shared/timing.py`. Offline, no API key, no cost.

Each step: read the short explanation, fill in the `# TODO` cell, run the check cell below it.

## Setup (just run it)

In [ ]:
from typing import Any

from docrag.llm.base import Provider
from docrag.llm.proxy import LLMProxy
from docrag.llm.types import ChatCompletion, Message, Usage
from docrag.shared import timing
from docrag.shared.cache import Cache

HELLO = [Message(role="user", content="hello")]

## Step 1: Your own provider

`Provider` is an abstract base class (Java `abstract class`). Its `chat()` is a *template method*: it calls `to_native()` (our messages -> the API's request), `_call()` (send it) and `from_native()` (the API's reply -> `ChatCompletion`). A subclass must implement all three, otherwise instantiating it raises `TypeError`. Write a provider that "answers" with the last user message in upper case.

In [ ]:
class ShoutProvider(Provider):
    name = "shout"

    # TODO 1: def to_native(self, model, messages, response_format, temperature) -> dict
    #         return {"text": messages[-1].text}
    # TODO 2: async def _call(self, native) -> return native["text"].upper()
    # TODO 3: def from_native(self, model, raw) -> ChatCompletion(model=model, content=raw,
    #         usage=Usage(input_tokens=1, output_tokens=1))

In [ ]:
try:
    provider = ShoutProvider()
except TypeError as err:
    raise AssertionError(f"Implement all three abstract methods first: {err}") from None
proxy = LLMProxy({"shout": provider})
completion = await proxy.chat("shout:any-model", HELLO)
assert completion.content == "HELLO", f"Expected 'HELLO', got {completion.content!r}"
print(completion)
print("OK")

## Step 2: Routes

Every call names a route `"provider:model"`. `proxy.resolve(route)` splits it on the **first** colon only (`str.partition`), because model names can contain colons themselves (`ollama:qwen3:8b`). Resolve the route below.

In [ ]:
# TODO: provider_obj, model = proxy.resolve(...) for the route "shout:qwen3:8b"
provider_obj, model = None, None

In [ ]:
assert provider_obj is provider, (
    "The route's provider part picks the registered provider"
)
assert model == "qwen3:8b", f"Only the FIRST colon splits; got {model!r}"
try:
    proxy.resolve("openai:gpt-6")
    raise AssertionError("an unregistered provider must fail")
except ValueError as err:
    print("clear error:", err)
print("OK")

## Step 3: A decorator that knows about cache hits

`@timing.timed("name")` wraps a function and records its duration (`timed("x")(func)` under the hood). A function that is answered from a cache calls `timing.cache_hit()`, and its duration goes to `name_cached` instead. Decorate `lookup` and call `timing.cache_hit()` on the cached path.

In [ ]:
memo: dict[str, str] = {}


# TODO: decorate with @timing.timed("lesson05") and call timing.cache_hit() when key is in memo
def lookup(key: str) -> str:
    if key in memo:
        return memo[key]
    memo[key] = key.upper()
    return memo[key]

In [ ]:
timing.reset()
lookup("a")
lookup("a")
lookup("a")
snap = timing.snapshot()
assert snap.get("lesson05", {}).get("n") == 1.0, (
    "One real computation -> 'lesson05' (did you decorate it?)"
)
assert snap.get("lesson05_cached", {}).get("n") == 2.0, (
    "Two cache hits -> 'lesson05_cached'"
)
# The proxy does the same with its LLM cache:
cached_proxy = LLMProxy({"shout": ShoutProvider()}, cache=Cache(":memory:"))
await cached_proxy.chat("shout:m", HELLO)
await cached_proxy.chat("shout:m", HELLO)
print({k: v["n"] for k, v in timing.snapshot().items()})
print("OK")

## Observation question

Why did timing the proxy *before* its cache report a wrong LLM latency?

<details>
<summary>Answer</summary>

`@timed` wraps the whole `chat()`, and the cache lookup happens inside it. On a re-run almost every call is a cache hit that takes a few milliseconds, so the recorded p50 for `llm` was the speed of a SQLite lookup (3 ms in a real `run.json`), not the speed of the model. Embedding and parsing had the same problem. Now cache hits call `timing.cache_hit()` and are recorded as `llm_cached` / `embed_cached` / `parse_cached`, so `llm` only contains real model calls.
</details>

## The real code

- Provider base class: [`src/docrag/llm/base.py`](../../src/docrag/llm/base.py)
- Proxy: [`src/docrag/llm/proxy.py`](../../src/docrag/llm/proxy.py)
- Fake provider: [`src/docrag/llm/fake.py`](../../src/docrag/llm/fake.py)
- @timed: [`src/docrag/shared/timing.py`](../../src/docrag/shared/timing.py)
- Lesson: [`docs/lessons/05_llm_proxy.md`](../../docs/lessons/05_llm_proxy.md)